<a id="network-meta-analysis-using-python"></a>

<h2 align="center" style="font-family:Times New Roman;font-variant: small-caps;">Network Meta-Analysis using Python</h2>

---

Data preparation using Python. 

**Directory structure** for organizing input files and output files.

```
.  
├── inputs/  
│   └── data/  
│       ├── extracted  
│       ├── reformatted  
│       └── sheets_for_analysis  
└── outputs/  \
    ├── plots  
    └── results
```

**Create directory structure**:

In [57]:
folders = {
    "input": f"./input/", # using Python (python.ipynb, this notebook)
        "data": f"./input/data/",
            "extracted": f"./input/data/extracted/",
            "reformatted": f"./input/data/reformatted/",
            "sheets_for_analysis": f"./input/data/sheets_for_analysis/",
    "output": f"./output/", # using R (R.ipynb)
        "plots": f"./output/plots/", # netgraph, forest
        "results": f"./output/results/", # netmeta, netleague, netrank
}

for name, path in folders.items():
    import os
    os.makedirs(path, exist_ok = True)
    globals()[name] = path


**Install libraries** and **import modules**

In [1]:
%%capture
%pip install pandas
%pip install openpyxl
%pip install tabulate
%pip install IPython
%pip install numpy

In [10]:
import pandas as pd
import openpyxl
import tabulate
from IPython.display import display, Markdown
import numpy as np

**Import** `master.xlsx` file.

In [9]:
data = pd.read_csv(f"{extracted}/master.csv", encoding = "utf-8")
data.head()

,Unnamed: 0,id,study,subgroup,composite_key,outcome,data_type,ni,yi,sdi,...,screening,age_mean,age_sd,total,missing,male,female,follow-up_mo,follow-up_yr,follow-up_term
0,0,1,"Ageberg et al., 2009",BPTB,"Ageberg et al., 2009, BPTB",tegner,continuous,16.0,4.17,1.41,...,included,32.0,5.00,20.0,4.0,15,5.0,36.0,3.0,medium
1,1,1,"Ageberg et al., 2009",HT,"Ageberg et al., 2009, HT",tegner,continuous,20.0,3.15,1.34,...,included,28.0,4.40,16.0,-4.0,12,4.0,36.0,3.0,medium
2,2,2,"Aglietti et al., 2004",BPTB,"Aglietti et al., 2004, BPTB",instrumented_laxity,continuous,60.0,2.20,1.08,...,included,25.0,4.97,60.0,0.0,46,14.0,24.0,2.0,medium
3,3,2,"Aglietti et al., 2004",HT,"Aglietti et al., 2004, HT",instrumented_laxity,continuous,60.0,1.95,1.30,...,included,25.0,5.19,60.0,0.0,46,14.0,24.0,2.0,medium
4,4,2,"Aglietti et al., 2004",BPTB,"Aglietti et al., 2004, BPTB",lachman_1,dichotomous,60.0,0.00,0.06,...,included,25.0,4.97,60.0,0.0,46,14.0,24.0,2.0,medium


In [11]:
included = data[data["screening"] == "included"]
excluded = data[data["screening"] == "excluded"]
data = included.drop(columns = ["screening"])
excluded.to_csv(f"{extracted}/excluded.csv", encoding = "utf-8")
data.to_csv(f"{extracted}/included.csv", encoding = "utf-8")

data.info()

<class 'pandas.DataFrame'>
Index: 425 entries, 0 to 611
Data columns (total 24 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Unnamed: 0      425 non-null    int64  
 1   id              425 non-null    int64  
 2   study           425 non-null    str    
 3   subgroup        425 non-null    str    
 4   composite_key   425 non-null    str    
 5   outcome         425 non-null    str    
 6   data_type       425 non-null    str    
 7   ni              425 non-null    float64
 8   yi              425 non-null    float64
 9   sdi             425 non-null    float64
 10  min             71 non-null     float64
 11  q1              4 non-null      float64
 12  median          32 non-null     float64
 13  q3              4 non-null      float64
 14  max             71 non-null     float64
 15  age_mean        398 non-null    float64
 16  age_sd          394 non-null    float64
 17  total           349 non-null    float64
 18  missin

**Separate** into individual dataframes by outcomes

In [13]:
outcomes = [
    "ikdc_subjective",
    "lysholm",
    "tegner",
    "instrumented_laxity",
    "pivot_shift_1",
    "pivot_shift_2",
    "lachman_1",
    "lachman_2",
    "graft_failure"
]

for outcome in outcomes:
    import pandas as pd
    df = data[data["outcome"] == f"{outcome}"]
    df["sei"] = (df["sdi"]/(df["ni"]**0.5)).round(3)
    df["ci.lb"] = (df["yi"] - (1.96*df["sei"])).round(3)
    df["ci.ub"] = (df["yi"] + (1.96*df["sei"])).round(3)
    
    df.to_csv(f"{extracted}/{outcome}.csv", encoding = "utf-8")
    df.to_excel(f"./data/extracted/xlsx/{outcome}.xlsx")
    globals()[f"{outcome}_extracted"] = df
    md = df.to_markdown()
    with open(f"{extracted}/{outcome}.md", "w") as f:
        f.write(md)
    k = len(df)
    print(f"{outcome} = {k} rows, {int(k/2)} studies") # two rows per study (k (number of studies) = rows/2) for extracted data

ikdc_subjective = 50 rows, 25 studies
lysholm = 64 rows, 32 studies
tegner = 44 rows, 22 studies
instrumented_laxity = 67 rows, 33 studies
pivot_shift_1 = 50 rows, 25 studies
pivot_shift_2 = 50 rows, 25 studies
lachman_1 = 38 rows, 19 studies
lachman_2 = 38 rows, 19 studies
graft_failure = 24 rows, 12 studies


In [60]:
extracted_dfs = [
    ikdc_subjective_extracted,
    lysholm_extracted,
    tegner_extracted,
    instrumented_laxity_extracted,
    pivot_shift_1_extracted,
    pivot_shift_2_extracted,
    lachman_1_extracted,
    lachman_2_extracted,
    graft_failure_extracted,
    
]

combined = pd.concat(extracted_dfs)
combined.to_csv(f"{data}/extracted.csv", encoding = "utf-8")

2. **Pivot the data** (one row per study)\
`./data/reformatted`

In [51]:
dfs = {
    "ikdc_subjective": ikdc_subjective_extracted,
    "lysholm": lysholm_extracted,
    "tegner": tegner_extracted,
    "instrumented_laxity": instrumented_laxity_extracted,
    "pivot_shift_1": pivot_shift_1_extracted,
    "pivot_shift_2": pivot_shift_2_extracted,
    "lachman_1": lachman_1_extracted,
    "lachman_2": lachman_2_extracted,
    "graft_failure": graft_failure_extracted,
    
}

for name, df in dfs.items():
    df = pd.read_csv(f"{extracted}/{name}.csv", encoding = "utf-8")
    df["group"] = df.groupby(["id"]).cumcount() + 1

    wide = (
        df.pivot(
            index= ["id", "study", "outcome"],
            columns="group",
            values=["subgroup", "ni", "yi", "sdi"]
        )
    )
    
    wide.columns = [
        f"{name}{group}" if name != "subgroup" else f"treat{group}"
        for name, group in wide.columns
    ]
    
    wide = (
        wide.rename(columns={
            "ni1": "n1",
            "yi1": "y1",
            "sdi1": "sd1",
            "ni2": "n2",
            "yi2": "y2",
            "sdi2": "sd2"
            
        })
        .reset_index()
    )

    integers = ["id", "n1", "n2"]
    numbers = ["y1", "y2", "sd1", "sd2"]

    for integer in integers:
        wide[integer] = pd.to_numeric(wide[integer]).astype("Int64")
    
    for number in numbers:
        wide[number] = pd.to_numeric(wide[number]).astype("Float64").round(3)
     
    import os
    os.makedirs(f"./data/reformatted/csv", exist_ok = True)
    wide.to_csv(f"{reformatted}/{name}.csv", encoding = "utf-8")
    
    os.makedirs(f"./data/reformatted/xlsx", exist_ok = True)
    wide.to_excel(f"./data/reformatted/xlsx/{name}.xlsx")
    globals()[f"{name}_reformatted"] = wide

Combined **reformatted** CSV file (wide-format)

In [61]:
reformatted_dfs = [
    ikdc_subjective_reformatted,
    lysholm_reformatted,
    tegner_reformatted,
    instrumented_laxity_reformatted,
    pivot_shift_1_reformatted,
    pivot_shift_2_reformatted,
    lachman_1_reformatted,
    lachman_2_reformatted,
    graft_failure_reformatted,
]

combined = pd.concat(reformatted_dfs)
combined.to_csv(f"{data}/reformatted.csv", encoding = "utf-8")

3. **Prepare sheets for analysis** (calculate standardized mean differences and standard errors of standardized mean differences as TE and seTE, respectively)\
`./data/sheets_for_analysis`

In [53]:
dfs = {
    "ikdc_subjective": ikdc_subjective_reformatted, 
    "lysholm": lysholm_reformatted, 
    "tegner": tegner_reformatted, 
    "instrumented_laxity": instrumented_laxity_reformatted,
    
}

for name, df in dfs.items():
    df = df.copy()
    
    def SMD(n1, n2, y1, y2, sd1, sd2):
        sd = (
            (((n1 - 1) * sd1**2) + ((n2 - 1) * sd2**2))
            / (n1 + n2 - 2)
        )
        sp = np.sqrt(sd)
        return round((np.abs(y1 - y2) / sp), 3)
    
    # Standard Error of the Standardized Mean Difference (SMD)
    def seSMD(n1, n2, smd):
        se = np.sqrt(
            (n1 + n2)/(n1*n2)
            + smd**2/(2*(n1+n2-2))
        )
        se = round(se, 3)
        return se

    df["TE"] = SMD(df["n1"], df["n2"], df["y1"], df["y2"], df["sd1"], df["sd2"])
    df["seTE"] = seSMD(df["n1"], df["n2"], df["TE"])

    df.to_csv(f"{sheets_for_analysis}/{name}.csv", encoding = "utf-8")
    df.to_excel(f"./data/sheets_for_analysis/xlsx/{name}.xlsx")
    df = df[["id", "study", "treat1", "treat2", "n1", "n2", "y1", "y2", "sd1", "sd2", "TE", "seTE"]]
    k = len(df)
    print(f"{name} = {k} rows, {int(k)} studies")
    
    globals()[f"{name}_analysis"] = df

ikdc_subjective = 25 rows, 25 studies
lysholm = 32 rows, 32 studies
tegner = 22 rows, 22 studies
instrumented_laxity = 34 rows, 34 studies


In [63]:
dfs = {
    "pivot_shift_1": pivot_shift_1_reformatted,
    "pivot_shift_2": pivot_shift_2_reformatted,
    "lachman_1": lachman_1_reformatted,
    "lachman_2": lachman_2_reformatted,
    "graft_failure": graft_failure_reformatted,
    
}

for name, df in dfs.items():
    df = df.copy()
    var = ["y1", "y2", "n1", "n2"]
    for v in var:
        df[f"{v}"] = df[f"{v}"] + 0.5
    df["TE"] = ((df["y1"]*(df["n2"] - df["y2"]))/(df["y2"]*(df["n1"] - df["y1"]))).round(3).fillna("")
    df["TE"] = df["TE"].round(3)
    
    df["seTE"] = ((1/df["y1"]) + (1/(df["n1"] - df["y1"])) + (1/df["y2"]) + (1/(df["n2"] - df["y2"])))**0.5
    df["seTE"] = df["seTE"].round(3)
    df.to_csv(f"{sheets_for_analysis}/{name}.csv", encoding = "utf-8")
    df = df[["id", "study", "treat1", "treat2", "n1", "n2", "y1", "y2", "sd1", "sd2", "TE", "seTE"]]
    k = len(df)
    print(f"{name} = {k} rows, {int(k)} studies")

    
    globals()[f"{name}_analysis"] = df

pivot_shift_1 = 25 rows, 25 studies
pivot_shift_2 = 25 rows, 25 studies
lachman_1 = 19 rows, 19 studies
lachman_2 = 19 rows, 19 studies
graft_failure = 12 rows, 12 studies


Combined **sheets for analysis** CSV file (wide-format with effect sizes: TE and seTE columns)

In [64]:
sheets_for_analysis_dfs = [
    ikdc_subjective_analysis,
    lysholm_analysis,
    tegner_analysis,
    instrumented_laxity_analysis,
    pivot_shift_1_analysis,
    pivot_shift_2_analysis,
    lachman_1_analysis,
    lachman_2_analysis,
    graft_failure_analysis
]

combined = pd.concat(sheets_for_analysis_dfs)
combined = combined.sort_values(by = "study", ascending = True)

combined.to_csv(f"{data}/sheets_for_analysis.csv", encoding = "utf-8")

**Display preview dataframes** (to check for correct columns for network meta-analysis)  

In [65]:
combined = {
    "extracted_data": extracted_dfs, 
    "reformatted_data": reformatted_dfs,
    "sheets_for_analysis_data": sheets_for_analysis_dfs
}

for name, dfs in combined.items():
    df = pd.concat(dfs)
    globals()[name] = df

**1. Extracted (long-format)**: one row per subgroup for each study.

In [66]:
extracted_data.head().fillna("")

,Unnamed: 0,id,study,subgroup,composite_key,outcome,data_type,ni,yi,sdi,...,total,missing,male,female,follow-up_mo,follow-up_yr,follow-up_term,sei,ci.lb,ci.ub
48,48,10,"Barié et al., 2020",BPTB,"Barié et al., 2020, BPTB",ikdc_subjective,continuous,43.0,91.0,7.3,...,30.0,-13.0,17,13.0,122.4,10.2,long,1.113,88.819,93.181
49,49,10,"Barié et al., 2020",QT,"Barié et al., 2020, QT",ikdc_subjective,continuous,43.0,92.0,11.5,...,30.0,-13.0,17,13.0,123.6,10.3,long,1.754,88.562,95.438
72,72,12,"Bi et al., 2018",HT,"Bi et al., 2018, HT",ikdc_subjective,continuous,62.0,90.4,7.1,...,62.0,0.0,31,31.0,30.3,2.5,medium,0.902,88.632,92.168
73,73,12,"Bi et al., 2018",PLT,"Bi et al., 2018, PLT",ikdc_subjective,continuous,62.0,89.3,8.4,...,62.0,0.0,34,28.0,29.7,2.5,medium,1.067,87.209,91.391
96,96,14,"Bottoni et al., 2015",HT,"Bottoni et al., 2015, HT",ikdc_subjective,continuous,44.0,79.5,24.7,...,48.0,4.0,41,7.0,120.0,10.0,long,3.724,72.201,86.799


**2. Reformatted (wide-format)**: one row per study (wide-format) with extracted data (sample size, effect size and standard deviation) for continuous and dichotomous outcomes.

In [67]:
reformatted_data.head()

,id,study,outcome,treat1,treat2,n1,n2,y1,y2,sd1,sd2
0,10,"Barié et al., 2020",ikdc_subjective,BPTB,QT,43,43,91.0,92.0,7.3,11.5
1,12,"Bi et al., 2018",ikdc_subjective,HT,PLT,62,62,90.4,89.3,7.1,8.4
2,14,"Bottoni et al., 2015",ikdc_subjective,HT,TA,44,36,79.5,81.1,24.7,23.1
3,18,"Butt et al., 2024",ikdc_subjective,HT,PLT,30,30,89.7,89.9,5.7,9.8
4,19,"Calvert et al., 2025",ikdc_subjective,HT,QT,92,89,92.0,89.0,8.0,10.0


**3. Sheets for analysis**: one row per study (wide-format) with effect size estimates (standardized mean difference and odds ratio for continuous and dichotomous outcomes, respectively) for each individual study.

In [68]:
sheets_for_analysis_data.head()

,id,study,treat1,treat2,n1,n2,y1,y2,sd1,sd2,TE,seTE
0,10,"Barié et al., 2020",BPTB,QT,43.0,43.0,91.0,92.0,7.3,11.5,0.104,0.216
1,12,"Bi et al., 2018",HT,PLT,62.0,62.0,90.4,89.3,7.1,8.4,0.141,0.18
2,14,"Bottoni et al., 2015",HT,TA,44.0,36.0,79.5,81.1,24.7,23.1,0.067,0.225
3,18,"Butt et al., 2024",HT,PLT,30.0,30.0,89.7,89.9,5.7,9.8,0.025,0.258
4,19,"Calvert et al., 2025",HT,QT,92.0,89.0,92.0,89.0,8.0,10.0,0.332,0.15


**4. Tables for manuscript** (in markdown for conversion to docx table using pandoc):\
\
**Table 1** Characteristics of included studies.\
**Table 2** Results of individual studies.\
**Table 3** Results of network meta-analysis.


**Table 1** Characteristics of included studies.

In [20]:
df = df_extracted.sort_values(by = ["id", "study", "subgroup"], ascending = True)
df1 = df[["id", "study", "subgroup", "outcome", "ni", "yi", "sdi"]]
#df1 = df.set_index(["id", "study", "subgroup", "outcome"])

df1["sei"] = df1["sdi"]/(df1["ni"]**0.5)
df1 = df1.rename(columns = {
    "ni": "ni",
    "yi": "TE",
    "sei": "seTE"
}).drop(columns = "sdi")
df1.info()

<class 'pandas.DataFrame'>
Index: 440 entries, 434 to 233
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   id        440 non-null    int64  
 1   study     440 non-null    str    
 2   subgroup  440 non-null    str    
 3   outcome   440 non-null    str    
 4   ni        440 non-null    float64
 5   TE        440 non-null    float64
 6   seTE      440 non-null    float64
dtypes: float64(3), int64(1), str(3)
memory usage: 27.5 KB


In [22]:
df = df_sheets_for_analysis.copy()
df2 = df_sheets_for_analysis.copy()
df2["ni"] = df["n1"] + df["n2"]
df2["subgroup"] = df["treat1"] + " vs. " + df["treat2"]
df2 = df2[["id", "study", "subgroup", "outcome", "ni", "TE", "seTE"]]

In [23]:
df2.info()

<class 'pandas.DataFrame'>
Index: 220 entries, 0 to 11
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   id        220 non-null    int64  
 1   study     220 non-null    str    
 2   subgroup  220 non-null    str    
 3   outcome   220 non-null    str    
 4   ni        220 non-null    float64
 5   TE        220 non-null    float64
 6   seTE      219 non-null    float64
dtypes: float64(3), int64(1), str(3)
memory usage: 13.8 KB


In [25]:
df3 = pd.concat([df1, df2])
df3 = df3.sort_values(by = "id")
df3.info()
df3.head()

<class 'pandas.DataFrame'>
Index: 660 entries, 434 to 122
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   id        660 non-null    int64  
 1   study     660 non-null    str    
 2   subgroup  660 non-null    str    
 3   outcome   660 non-null    str    
 4   ni        660 non-null    float64
 5   TE        660 non-null    float64
 6   seTE      659 non-null    float64
dtypes: float64(3), int64(1), str(3)
memory usage: 41.2 KB


**Table 2** Results of individual studies.

Need:

| Study | Subgroup | Outcome | N | TE | seTE |
|-------|----------|---------|---|----|------|
| 1     | A        | ikdc    |   | y1 |      |
| 1     | B        | ikdc    |   | y2 |      |
| 1     | A vs. B  | ikdc    |   | TE |      |

In [34]:
df3.info()
df3["ci.lb"] = df3["TE"] - (1.96 * df3["seTE"])
df3["ci.ub"] = df3["TE"] + (1.96 * df3["seTE"])

print(df3.columns)
columns = ['TE', 'seTE', 'ci.lb','ci.ub']
for column in columns:
    df3[column] = df3[column].round(3)
    
df3.head()
df3.sort_values(by = "id", ascending = True)
df3.to_csv("./data/tables/csv/results.csv", encoding = "utf-8")
df3.to_excel("./data/tables/xlsx/results.xlsx")

<class 'pandas.DataFrame'>
Index: 660 entries, 434 to 122
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   id        660 non-null    int64  
 1   study     660 non-null    str    
 2   subgroup  660 non-null    str    
 3   outcome   660 non-null    str    
 4   ni        660 non-null    float64
 5   TE        660 non-null    float64
 6   seTE      659 non-null    float64
 7   ci.lb     658 non-null    float64
 8   ci.ub     659 non-null    float64
dtypes: float64(5), int64(1), str(3)
memory usage: 51.6 KB
Index(['id', 'study', 'subgroup', 'outcome', 'ni', 'TE', 'seTE', 'ci.lb',
       'ci.ub'],
      dtype='str')


In [56]:
results = pd.DataFrame({
    "id": df["id"],
    "study": df["study"],
    "subgroup": df["treat1"] + " vs. " + df["treat2"],
    "ni": df["n1"] + df["n2"],
    "TE": df["TE"],
    "seTE": df["seTE"],
    "male": df["male1"] + df["male2"],
    "female": df["female1"] + df["female2"],
    "follow-up_mo": df["follow-up_mo1"],
    "follow-up_yr": df["follow-up_yr1"],
})

In [39]:
markdown = master_analysis.to_markdown()
with open("./data/sheets_for_analysis/master.md", "w") as f:
    f.write(f"{markdown}")
display(Markdown(markdown))

|                                                            | treat1   | treat2   |    n1 |    n2 |    y1 |    y2 |   sd1 |   sd2 |   age_mean1 |   age_mean2 |   age_sd1 |   age_sd2 |   total1 |   total2 | male1   |   male2 |   female1 |   female2 |   follow-up_mo1 |   follow-up_mo2 |   follow-up_yr1 |   follow-up_yr2 |      TE |       seTE |
|:-----------------------------------------------------------|:---------|:---------|------:|------:|------:|------:|------:|------:|------------:|------------:|----------:|----------:|---------:|---------:|:--------|--------:|----------:|----------:|----------------:|----------------:|----------------:|----------------:|--------:|-----------:|
| (1, 'Ageberg et al., 2009', 'tegner')                      | BPTB     | HT       |  16   |  20   |  4.17 |  3.15 |  1.41 |  1.34 |       32    |       28    |      5    |      4.4  |       20 |       16 | 15      |      12 |         5 |         4 |           36    |           36    |             3   |             3   |   0.744 |   0.347    |
| (2, 'Aglietti et al., 2004', 'instrumented_laxity')        | BPTB     | HT       |  60   |  60   |  2.2  |  1.95 |  1.08 |  1.3  |       25    |       25    |      4.97 |      5.19 |       60 |       60 | 46      |      46 |        14 |        14 |           24    |           24    |             2   |             2   |   0.209 |   0.183    |
| (2, 'Aglietti et al., 2004', 'lachman_1')                  | BPTB     | HT       |  60.5 |  60.5 |  0.5  |  0.5  |  0.06 |  0.06 |       25    |       25    |      4.97 |      5.19 |       60 |       60 | 46      |      46 |        14 |        14 |           24    |           24    |             2   |             2   |   1     |   2.00832  |
| (2, 'Aglietti et al., 2004', 'pivot_shift_2')              | BPTB     | HT       |  60.5 |  60.5 |  0.5  |  0.5  |  0.07 |  0.07 |       25    |       25    |      4.97 |      5.19 |       60 |       60 | 46      |      46 |        14 |        14 |           24    |           24    |             2   |             2   |   1     |   2.00832  |
| (2, 'Aglietti et al., 2004', 'pivot_shift_1')              | BPTB     | HT       |  60.5 |  60.5 | 10.5  | 11.5  |  0.07 |  0.07 |       25    |       25    |      4.97 |      5.19 |       60 |       60 | 46      |      46 |        14 |        14 |           24    |           24    |             2   |             2   |   0.895 |   0.471808 |
| (2, 'Aglietti et al., 2004', 'lachman_2')                  | BPTB     | HT       |  60.5 |  60.5 |  0.5  |  0.5  |  0.06 |  0.06 |       25    |       25    |      4.97 |      5.19 |       60 |       60 | 46      |      46 |        14 |        14 |           24    |           24    |             2   |             2   |   1     |   2.00832  |
| (3, 'Ahldén et al., 2009', 'lysholm')                      | BPTB     | HT       |  25   |  22   | 80.7  | 79.16 | 21.64 | 13.35 |       27.41 |       28.6  |      8.9  |      6.36 |       22 |       25 | 14      |      18 |         8 |         7 |           90.27 |           86.79 |             7.5 |             7.2 |   0.084 |   0.292    |
| (3, 'Ahldén et al., 2009', 'tegner')                       | BPTB     | HT       |  25   |  22   |  4.87 |  4.86 |  2.29 |  1.31 |       27.41 |       28.6  |      8.9  |      6.36 |       22 |       25 | 14      |      18 |         8 |         7 |           90.27 |           86.79 |             7.5 |             7.2 |   0.005 |   0.292    |
| (3, 'Ahldén et al., 2009', 'instrumented_laxity')          | BPTB     | HT       |  25   |  22   |  2.6  |  1.4  |  3.3  |  2.6  |       27.41 |       28.6  |      8.9  |      6.36 |       22 |       25 | 14      |      18 |         8 |         7 |           90.27 |           86.79 |             7.5 |             7.2 |   0.401 |   0.295    |
| (5, 'Anderson et al., 2001', 'instrumented_laxity')        | BPTB     | HT       |  35   |  33   |  2.1  |  3.1  |  2    |  2.3  |       23.6  |       21.02 |      7.13 |      5.81 |       35 |       68 | 23      |      45 |        12 |        23 |           34.6  |           35.8  |             2.9 |             3   |   0.465 |   0.246    |
| (5, 'Anderson et al., 2001', 'pivot_shift_2')              | BPTB     | HT       |  35.5 |  33.5 | 15.5  | 27.5  |  1.59 |  1.95 |       23.6  |       21.02 |      7.13 |      5.81 |       35 |       68 | 23      |      45 |        12 |        23 |           34.6  |           35.8  |             2.9 |             3   |   0.169 |   0.563513 |
| (5, 'Anderson et al., 2001', 'pivot_shift_1')              | BPTB     | HT       |  35.5 |  33.5 | 18.5  | 36.5  |  1.59 |  1.95 |       23.6  |       21.02 |      7.13 |      5.81 |       35 |       68 | 23      |      45 |        12 |        23 |           34.6  |           35.8  |             2.9 |             3   |  -0.089 | nan        |
| (6, 'Anz et al., 2023', 'graft_failure')                   | BPTB     | HT       |   8.5 |   8.5 |  0.5  |  0.5  |  0.18 |  0.18 |       29.4  |       27.55 |      7.34 |      8.64 |       10 |       10 | 4       |       3 |         6 |         7 |           12    |           12    |             1   |             1   |   1     |   2.06155  |
| (7, 'Aune et al., 2001', 'instrumented_laxity')            | BPTB     | HT       |  32   |  29   |  3.3  |  2.9  |  2.6  |  2.4  |       25    |       27    |      7    |      9    |       35 |       37 | 19      |      21 |        16 |        16 |           24    |           24    |             2   |             2   |   0.16  |   0.257    |
| (9, 'Barenius et al., 2010', 'lysholm')                    | BPTB     | HT       |  78   |  75   | 85.82 | 86.42 | 11.83 | 12.53 |       33    |       35    |      6.3  |      7.5  |       78 |       75 | 38      |      51 |        40 |        24 |           96    |           96    |             8   |             8   |   0.049 |   0.162    |
| (9, 'Barenius et al., 2010', 'tegner')                     | BPTB     | HT       |  78   |  75   |  5.8  |  4.07 |  1.87 |  1.88 |       33    |       35    |      6.3  |      7.5  |       78 |       75 | 38      |      51 |        40 |        24 |           96    |           96    |             8   |             8   |   0.923 |   0.17     |
| (10, 'Barié et al., 2020', 'tegner')                       | BPTB     | QT       |  43   |  43   |  6.1  |  6.1  |  1.14 |  1.14 |       30.6  |       30.5  |      7.5  |      7.8  |       30 |       30 | 17      |      17 |        13 |        13 |          122.4  |          123.6  |            10.2 |            10.3 |   0     |   0.216    |
| (10, 'Barié et al., 2020', 'lysholm')                      | BPTB     | QT       |  43   |  43   | 95.2  | 95.6  |  6.6  |  7.8  |       30.6  |       30.5  |      7.5  |      7.8  |       30 |       30 | 17      |      17 |        13 |        13 |          122.4  |          123.6  |            10.2 |            10.3 |   0.055 |   0.216    |
| (10, 'Barié et al., 2020', 'ikdc_subjective')              | BPTB     | QT       |  43   |  43   | 91    | 92    |  7.3  | 11.5  |       30.6  |       30.5  |      7.5  |      7.8  |       30 |       30 | 17.0    |      17 |        13 |        13 |          122.4  |          123.6  |            10.2 |            10.3 |   0.104 |   0.216    |
| (10, 'Barié et al., 2020', 'instrumented_laxity')          | BPTB     | QT       |  43   |  43   |  1.05 |  1    |  1.36 |  1.1  |       30.6  |       30.5  |      7.5  |      7.8  |       30 |       30 | 17      |      17 |        13 |        13 |          122.4  |          123.6  |            10.2 |            10.3 |   0.04  |   0.216    |
| (11, 'Beynnon et al., 2002', 'pivot_shift_1')              | BPTB     | HT       |  22.5 |  22.5 |  3.5  |  9.5  |  0.11 |  1.91 |       28.5  |       29.9  |      6.96 |      5.72 |       28 |       28 | 18      |      13 |        10 |        15 |           36    |           36    |             3   |             3   |   0.252 |   0.721479 |
| (11, 'Beynnon et al., 2002', 'pivot_shift_2')              | BPTB     | HT       |  22.5 |  22.5 |  6.5  |  1.5  |  0.11 |  1.91 |       28.5  |       29.9  |      6.96 |      5.72 |       28 |       28 | 18      |      13 |        10 |        15 |           36    |           36    |             3   |             3   |   5.688 |   0.964693 |
| (11, 'Beynnon et al., 2002', 'lachman_1')                  | BPTB     | HT       |  22.5 |  22.5 | 13.5  | 20.5  |  0.59 |  7.04 |       28.5  |       29.9  |      6.96 |      5.72 |       28 |       28 | 18      |      13 |        10 |        15 |           36    |           36    |             3   |             3   |   0.146 |   0.856718 |
| (11, 'Beynnon et al., 2002', 'lachman_2')                  | BPTB     | HT       |  22.5 |  22.5 | 13.5  |  7.5  |  0.59 |  7.04 |       28.5  |       29.9  |      6.96 |      5.72 |       28 |       28 | 18      |      13 |        10 |        15 |           36    |           36    |             3   |             3   |   3     |   0.620633 |
| (12, 'Bi et al., 2018', 'pivot_shift_1')                   | HT       | PLT      |  62.5 |  62.5 |  4.5  |  5.5  |  0.06 |  0.06 |       27.9  |       29.1  |      6.7  |      6.5  |       62 |       62 | 31      |      34 |        31 |        28 |           30.3  |           29.7  |             2.5 |             2.5 |   0.804 |   0.662439 |
| (12, 'Bi et al., 2018', 'pivot_shift_2')                   | HT       | PLT      |  62.5 |  62.5 |  1.5  |  0.5  |  0.06 |  0.06 |       27.9  |       29.1  |      6.7  |      6.5  |       62 |       62 | 31      |      34 |        31 |        28 |           30.3  |           29.7  |             2.5 |             2.5 |   3.049 |   1.64292  |
| (12, 'Bi et al., 2018', 'instrumented_laxity')             | HT       | PLT      |  62   |  62   |  1.71 |  1.85 |  0.57 |  0.77 |       27.9  |       29.1  |      6.7  |      6.5  |       62 |       62 | 31      |      34 |        31 |        28 |           30.3  |           29.7  |             2.5 |             2.5 |   0.207 |   0.18     |
| (12, 'Bi et al., 2018', 'ikdc_subjective')                 | HT       | PLT      |  62   |  62   | 90.4  | 89.3  |  7.1  |  8.4  |       27.9  |       29.1  |      6.7  |      6.5  |       62 |       62 | 31.0    |      34 |        31 |        28 |           30.3  |           29.7  |             2.5 |             2.5 |   0.141 |   0.18     |
| (13, 'Björnsson et al., 2016', 'ikdc_subjective')          | BPTB     | HT       |  61   |  86   | 67.3  | 74    | 20.8  | 18.8  |       28.2  |       26.8  |      9.1  |      7.6  |       61 |       86 | 42.0    |      53 |        19 |        33 |          202.6  |          191.9  |            16.9 |            16   |   0.341 |   0.169    |
| (13, 'Björnsson et al., 2016', 'tegner')                   | BPTB     | HT       |  61   |  86   |  4.1  |  4    |  1.7  |  1.7  |       28.2  |       26.8  |      9.1  |      7.6  |       61 |       86 | 42      |      53 |        19 |        33 |          202.6  |          191.9  |            16.9 |            16   |   0.059 |   0.167    |
| (13, 'Björnsson et al., 2016', 'lysholm')                  | BPTB     | HT       |  61   |  86   | 79.4  | 80.7  | 16.9  | 15.3  |       28.2  |       26.8  |      9.1  |      7.6  |       61 |       86 | 42      |      53 |        19 |        33 |          202.6  |          191.9  |            16.9 |            16   |   0.081 |   0.167    |
| (13, 'Björnsson et al., 2016', 'pivot_shift_2')            | BPTB     | HT       |  61.5 |  86.5 | 12.5  | 13.5  |  1.99 |  1.72 |       28.2  |       26.8  |      9.1  |      7.6  |       61 |       86 | 42      |      53 |        19 |        33 |          202.6  |          191.9  |            16.9 |            16   |   1.379 |   0.433798 |
| (13, 'Björnsson et al., 2016', 'lachman_2')                | BPTB     | HT       |  61.5 |  86.5 |  7.5  | 21.5  |  1.09 |  2.88 |       28.2  |       26.8  |      9.1  |      7.6  |       61 |       86 | 42      |      53 |        19 |        33 |          202.6  |          191.9  |            16.9 |            16   |   0.42  |   0.462329 |
| (13, 'Björnsson et al., 2016', 'pivot_shift_1')            | BPTB     | HT       |  61.5 |  86.5 | 26.5  | 28.5  |  1.99 |  1.72 |       28.2  |       26.8  |      9.1  |      7.6  |       61 |       86 | 42      |      53 |        19 |        33 |          202.6  |          191.9  |            16.9 |            16   |   1.541 |   0.344436 |
| (13, 'Björnsson et al., 2016', 'lachman_1')                | BPTB     | HT       |  61.5 |  86.5 | 31.5  | 68.5  |  1.09 |  2.88 |       28.2  |       26.8  |      9.1  |      7.6  |       61 |       86 | 42      |      53 |        19 |        33 |          202.6  |          191.9  |            16.9 |            16   |   0.276 |   0.367741 |
| (14, 'Bottoni et al., 2015', 'ikdc_subjective')            | HT       | TA       |  44   |  36   | 79.5  | 81.1  | 24.7  | 23.1  |       28.9  |       29.2  |      5.8  |      5.5  |       48 |       49 | 41.0    |      43 |         7 |         6 |          120    |          120    |            10   |            10   |   0.067 |   0.225    |
| (14, 'Bottoni et al., 2015', 'tegner')                     | HT       | TA       |  44   |  36   |  5.1  |  5.1  |  2.2  |  2    |       28.9  |       29.2  |      5.8  |      5.5  |       48 |       49 | 41      |      43 |         7 |         6 |          120    |          120    |            10   |            10   |   0     |   0.225    |
| (18, 'Butt et al., 2024', 'lysholm')                       | HT       | PLT      |  30   |  30   | 95.1  | 94.5  |  5.2  |  6.4  |       29.17 |       27.73 |      5.03 |      4.14 |       30 |       30 | 29      |      30 |         1 |         0 |           60    |           60    |             5   |             5   |   0.103 |   0.258    |
| (18, 'Butt et al., 2024', 'ikdc_subjective')               | HT       | PLT      |  30   |  30   | 89.7  | 89.9  |  5.7  |  9.8  |       29.17 |       27.73 |      5.03 |      4.14 |       30 |       30 | 29.0    |      30 |         1 |         0 |           60    |           60    |             5   |             5   |   0.025 |   0.258    |
| (19, 'Calvert et al., 2025', 'lysholm')                    | HT       | QT       |  92   |  89   | 94    | 93    |  7    |  8    |       29.4  |       28.1  |      7.7  |      8.2  |       55 |       57 | 28      |      28 |        27 |        29 |           24    |           24    |             2   |             2   |   0.133 |   0.149    |
| (19, 'Calvert et al., 2025', 'ikdc_subjective')            | HT       | QT       |  92   |  89   | 92    | 89    |  8    | 10    |       29.4  |       28.1  |      7.7  |      8.2  |       55 |       57 | 28.0    |      28 |        27 |        29 |           24    |           24    |             2   |             2   |   0.332 |   0.15     |
| (23, 'Ejerhed et al., 2003', 'lachman_2')                  | BPTB     | HT       |  32.5 |  34.5 |  2.5  |  0.5  |  5.16 |  4.23 |       27.26 |       30.77 |      8.47 |     10.52 |       32 |       34 | 21      |      25 |        11 |         9 |           24    |           24    |             2   |             2   |   5.667 |   1.56931  |
| (23, 'Ejerhed et al., 2003', 'lachman_1')                  | BPTB     | HT       |  32.5 |  34.5 | 14.5  | 18.5  |  5.16 |  4.23 |       27.26 |       30.77 |      8.47 |     10.52 |       32 |       34 | 21      |      25 |        11 |         9 |           24    |           24    |             2   |             2   |   0.697 |   0.490994 |
| (23, 'Ejerhed et al., 2003', 'tegner')                     | BPTB     | HT       |  32   |  34   |  5.77 |  6.39 |  1.94 |  1.43 |       27.26 |       30.77 |      8.47 |     10.52 |       32 |       34 | 21      |      25 |        11 |         9 |           24    |           24    |             2   |             2   |   0.366 |   0.248    |
| (23, 'Ejerhed et al., 2003', 'instrumented_laxity')        | BPTB     | HT       |  32   |  34   |  2.29 |  2.47 |  3.99 |  3.47 |       27.26 |       30.77 |      8.47 |     10.52 |       32 |       34 | 21      |      25 |        11 |         9 |           24    |           24    |             2   |             2   |   0.048 |   0.246    |
| (23, 'Ejerhed et al., 2003', 'lysholm')                    | BPTB     | HT       |  32   |  34   | 89.96 | 86.79 | 13.06 | 11.72 |       27.26 |       30.77 |      8.47 |     10.52 |       32 |       34 | 21      |      25 |        11 |         9 |           24    |           24    |             2   |             2   |   0.256 |   0.247    |
| (25, 'Eriksson et al., 2001', 'instrumented_laxity')       | BPTB     | HT       |  42   |  47   |  0.95 |  1.1  |  0.91 |  0.95 |       27    |       27    |      6    |      6    |      nan |      nan | nan     |     nan |       nan |       nan |            6.7  |            6.7  |             0.6 |             0.6 |   0.161 |   0.213    |
| (25, 'Eriksson et al., 2001', 'pivot_shift_2')             | BPTB     | HT       |  42.5 |  47.5 |  0.5  |  0.5  |  0.41 |  1.71 |       27    |       27    |      6    |      6    |      nan |      nan | nan     |     nan |       nan |       nan |            6.7  |            6.7  |             0.6 |             0.6 |   1.119 |   2.01124  |
| (25, 'Eriksson et al., 2001', 'pivot_shift_1')             | BPTB     | HT       |  42.5 |  47.5 |  6.5  | 19.5  |  0.41 |  1.71 |       27    |       27    |      6    |      6    |      nan |      nan | nan     |     nan |       nan |       nan |            6.7  |            6.7  |             0.6 |             0.6 |   0.259 |   0.518286 |
| (26, 'Feller & Webster, 2003', 'pivot_shift_2')            | BPTB     | HT       |  27.5 |  21.5 |  0.5  |  0.5  |  0.1  |  1.58 |       25.8  |       26.3  |      6    |      6    |       31 |       34 | 23      |      24 |         8 |        10 |           36    |           36    |             3   |             3   |   0.778 |   2.02105  |
| (26, 'Feller & Webster, 2003', 'lachman_1')                | BPTB     | HT       |  27.5 |  21.5 |  6.5  | 19.5  |  0.3  |  0.11 |       25.8  |       26.3  |      6    |      6    |       31 |       34 | 23      |      24 |         8 |        10 |           36    |           36    |             3   |             3   |   0.032 |   0.86761  |
| (26, 'Feller & Webster, 2003', 'lachman_2')                | BPTB     | HT       |  27.5 |  21.5 |  0.5  |  0.5  |  0.3  |  0.11 |       25.8  |       26.3  |      6    |      6    |       31 |       34 | 23      |      24 |         8 |        10 |           36    |           36    |             3   |             3   |   0.778 |   2.02105  |
| (26, 'Feller & Webster, 2003', 'pivot_shift_1')            | BPTB     | HT       |  27.5 |  21.5 |  2.5  |  9.5  |  0.1  |  1.58 |       25.8  |       26.3  |      6    |      6    |       31 |       34 | 23      |      24 |         8 |        10 |           36    |           36    |             3   |             3   |   0.126 |   0.792841 |
| (26, 'Feller & Webster, 2003', 'instrumented_laxity')      | BPTB     | HT       |  27   |  21   |  0.5  |  1.1  |  1    |  1.2  |       25.8  |       26.3  |      6    |      6    |       31 |       34 | 23      |      24 |         8 |        10 |           36    |           36    |             3   |             3   |   0.55  |   0.297    |
| (30, 'Gifstad et al., 2013', 'instrumented_laxity')        | BPTB     | HT       |  41   |  36   |  1.4  |  1.4  |  1.8  |  1.4  |       27    |       27    |      6.74 |      6.78 |      nan |      nan | nan     |     nan |       nan |       nan |           84    |           84    |             7   |             7   |   0     |   0.228    |
| (30, 'Gifstad et al., 2013', 'lachman_1')                  | BPTB     | HT       |  41.5 |  36.5 | 16.5  | 13.5  |  0.24 |  0.26 |       27    |       27    |      6.74 |      6.78 |      nan |      nan | nan     |     nan |       nan |       nan |           84    |           84    |             7   |             7   |   1.124 |   0.467074 |
| (30, 'Gifstad et al., 2013', 'lachman_2')                  | BPTB     | HT       |  41.5 |  36.5 |  1.5  |  1.5  |  0.24 |  0.26 |       27    |       27    |      6.74 |      6.78 |      nan |      nan | nan     |     nan |       nan |       nan |           84    |           84    |             7   |             7   |   0.875 |   1.17767  |
| (32, 'Guglielmetti et al., 2021', 'lachman_2')             | BPTB     | HT       |  31.5 |  31.5 |  2.5  |  3.5  |  0.48 |  0.7  |       25.2  |       24.64 |      5.48 |      5.33 |       31 |       31 | 23      |      19 |         8 |        12 |           24    |           24    |             2   |             2   |   0.69  |   0.869432 |
| (32, 'Guglielmetti et al., 2021', 'pivot_shift_2')         | BPTB     | HT       |  31.5 |  31.5 |  2.5  |  1.5  |  0.48 |  0.7  |       25.2  |       24.64 |      5.48 |      5.33 |       31 |       31 | 23      |      19 |         8 |        12 |           24    |           24    |             2   |             2   |   1.724 |   1.06512  |
| (32, 'Guglielmetti et al., 2021', 'pivot_shift_1')         | BPTB     | HT       |  31.5 |  31.5 |  7.5  | 10.5  |  0.48 |  0.7  |       25.2  |       24.64 |      5.48 |      5.33 |       31 |       31 | 23      |      19 |         8 |        12 |           24    |           24    |             2   |             2   |   0.625 |   0.563788 |
| (32, 'Guglielmetti et al., 2021', 'lachman_1')             | BPTB     | HT       |  31.5 |  31.5 |  9.5  |  9.5  |  0.48 |  0.7  |       25.2  |       24.64 |      5.48 |      5.33 |       31 |       31 | 23      |      19 |         8 |        12 |           24    |           24    |             2   |             2   |   1     |   0.549031 |
| (32, 'Guglielmetti et al., 2021', 'lysholm')               | BPTB     | HT       |  31   |  31   | 89.87 | 91.26 |  9.17 |  9.36 |       25.2  |       24.64 |      5.48 |      5.33 |       31 |       31 | 23      |      19 |         8 |        12 |           24    |           24    |             2   |             2   |   0.15  |   0.254    |
| (32, 'Guglielmetti et al., 2021', 'ikdc_subjective')       | BPTB     | HT       |  31   |  31   | 88.75 | 91    |  9.27 |  9.07 |       25.2  |       24.64 |      5.48 |      5.33 |       31 |       31 | 23.0    |      19 |         8 |        12 |           24    |           24    |             2   |             2   |   0.245 |   0.255    |
| (33, 'Harilainen et al., 2006', 'lachman_2')               | BPTB     | HT       |  37.5 |  37.5 |  0.5  |  1.5  |  0.08 |  0.25 |      nan    |      nan    |    nan    |    nan    |      nan |      nan | nan     |     nan |       nan |       nan |           60.6  |           60.6  |             5.1 |             5.1 |   0.324 |   1.64969  |
| (33, 'Harilainen et al., 2006', 'pivot_shift_1')           | BPTB     | HT       |  37.5 |  37.5 |  4.5  |  4.5  |  0.08 |  0.25 |      nan    |      nan    |    nan    |    nan    |      nan |      nan | nan     |     nan |       nan |       nan |           60.6  |           60.6  |             5.1 |             5.1 |   1     |   0.710669 |
| (33, 'Harilainen et al., 2006', 'instrumented_laxity')     | BPTB     | HT       |  37   |  37   |  1.5  |  1.2  |  3    |  3.9  |      nan    |      nan    |    nan    |    nan    |      nan |      nan | nan     |     nan |       nan |       nan |           60.6  |           60.6  |             5.1 |             5.1 |   0.086 |   0.233    |
| (33, 'Harilainen et al., 2006', 'lachman_1')               | BPTB     | HT       |  37.5 |  37.5 |  8.5  |  6.5  |  0.08 |  0.25 |      nan    |      nan    |    nan    |    nan    |      nan |      nan | nan     |     nan |       nan |       nan |           60.6  |           60.6  |             5.1 |             5.1 |   1.398 |   0.581579 |
| (33, 'Harilainen et al., 2006', 'lysholm')                 | BPTB     | HT       |  37   |  37   | 89.11 | 95    | 15.53 | 15.53 |      nan    |      nan    |    nan    |    nan    |      nan |      nan | nan     |     nan |       nan |       nan |           60.6  |           60.6  |             5.1 |             5.1 |   0.379 |   0.235    |
| (33, 'Harilainen et al., 2006', 'pivot_shift_2')           | BPTB     | HT       |  37.5 |  37.5 |  0.5  |  1.5  |  0.08 |  0.25 |      nan    |      nan    |    nan    |    nan    |      nan |      nan | nan     |     nan |       nan |       nan |           60.6  |           60.6  |             5.1 |             5.1 |   0.324 |   1.64969  |
| (33, 'Harilainen et al., 2006', 'tegner')                  | BPTB     | HT       |  37   |  37   |  5.89 |  6    |  2.12 |  1.41 |      nan    |      nan    |    nan    |    nan    |      nan |      nan | nan     |     nan |       nan |       nan |           60.6  |           60.6  |             5.1 |             5.1 |   0.061 |   0.233    |
| (34, 'Heijne & Werner, 2010', 'tegner')                    | BPTB     | HT       |  30   |  29   |  6.64 |  5    |  2.21 |  1.48 |       29    |       30    |      7    |      9    |       34 |       34 | 22      |      14 |        12 |        20 |           24.7  |           24.7  |             2.1 |             2.1 |   0.869 |   0.273    |
| (34, 'Heijne & Werner, 2010', 'pivot_shift_2')             | BPTB     | HT       |  30.5 |  29.5 |  3.5  |  8.5  |  0.71 |  2.18 |       29    |       30    |      7    |      9    |       34 |       34 | 22      |      14 |        12 |        20 |           24.7  |           24.7  |             2.1 |             2.1 |   0.32  |   0.698582 |
| (34, 'Heijne & Werner, 2010', 'pivot_shift_1')             | BPTB     | HT       |  30.5 |  29.5 | 13.5  | 22.5  |  0.71 |  2.18 |       29    |       30    |      7    |      9    |       34 |       34 | 22      |      14 |        12 |        20 |           24.7  |           24.7  |             2.1 |             2.1 |   0.247 |   0.565861 |
| (35, 'Holm et al., 2010', 'lysholm')                       | BPTB     | HT       |  28   |  29   | 84.2  | 86.1  | 15.4  | 15.1  |       33.9  |       37.9  |      6.2  |      9.1  |       28 |       29 | 18      |      15 |        10 |        14 |           10.2  |           10.7  |             0.9 |             0.9 |   0.125 |   0.265    |
| (35, 'Holm et al., 2010', 'instrumented_laxity')           | BPTB     | HT       |  28   |  29   |  3    |  2    |  3.2  |  3.5  |       33.9  |       37.9  |      6.2  |      9.1  |       28 |       29 | 18      |      15 |        10 |        14 |           10.2  |           10.7  |             0.9 |             0.9 |   0.298 |   0.266    |
| (36, 'Horstmann et al., 2022', 'ikdc_subjective')          | HT       | QT       |  26   |  23   | 83.7  | 89.3  | 12.7  | 12.2  |       32.7  |       24.1  |     11.4  |      3.6  |       27 |       24 | 12.0    |      21 |        15 |         3 |           24    |           24    |             2   |             2   |   0.449 |   0.29     |
| (36, 'Horstmann et al., 2022', 'pivot_shift_2')            | HT       | QT       |  26.5 |  23.5 |  0.5  |  0.5  |  0.1  |  0.1  |       32.7  |       24.1  |     11.4  |      3.6  |       27 |       24 | 12      |      21 |        15 |         3 |           24    |           24    |             2   |             2   |   0.885 |   2.02038  |
| (36, 'Horstmann et al., 2022', 'lysholm')                  | HT       | QT       |  26   |  23   | 83.5  | 90.4  | 17.4  | 11.9  |       32.7  |       24.1  |     11.4  |      3.6  |       27 |       24 | 12      |      21 |        15 |         3 |           24    |           24    |             2   |             2   |   0.458 |   0.29     |
| (36, 'Horstmann et al., 2022', 'instrumented_laxity')      | HT       | QT       |  26   |  23   |  0.2  |  0.7  |  2.2  |  1.1  |       32.7  |       24.1  |     11.4  |      3.6  |       27 |       24 | 12      |      21 |        15 |         3 |           24    |           24    |             2   |             2   |   0.282 |   0.288    |
| (36, 'Horstmann et al., 2022', 'pivot_shift_1')            | HT       | QT       |  26.5 |  23.5 |  0.5  |  0.5  |  0.1  |  0.1  |       32.7  |       24.1  |     11.4  |      3.6  |       27 |       24 | 12      |      21 |        15 |         3 |           24    |           24    |             2   |             2   |   0.885 |   2.02038  |
| (37, 'Ibrahim et al., 2005', 'lysholm')                    | BPTB     | HT       |  40   |  46   | 91.6  | 92.7  |  3.94 |  3.62 |       22.3  |       22.3  |      3.94 |      3.84 |       40 |       46 | 40      |      46 |         0 |         0 |           81    |           81    |             6.8 |             6.8 |   0.292 |   0.217    |
| (37, 'Ibrahim et al., 2005', 'lachman_1')                  | BPTB     | HT       |  40.5 |  46.5 |  5.5  |  7.5  |  0.99 |  1.3  |       22.3  |       22.3  |      3.94 |      3.84 |       40 |       46 | 40      |      46 |         0 |         0 |           81    |           81    |             6.8 |             6.8 |   0.817 |   0.607753 |
| (37, 'Ibrahim et al., 2005', 'pivot_shift_2')              | BPTB     | HT       |  40.5 |  45.5 |  0.5  |  0.5  |  0.99 |  1.54 |       22.3  |       22.3  |      3.94 |      3.84 |       40 |       46 | 40      |      46 |         0 |         0 |           81    |           81    |             6.8 |             6.8 |   1.125 |   2.01177  |
| (37, 'Ibrahim et al., 2005', 'lachman_2')                  | BPTB     | HT       |  40.5 |  46.5 |  0.5  |  0.5  |  0.99 |  1.3  |       22.3  |       22.3  |      3.94 |      3.84 |       40 |       46 | 40      |      46 |         0 |         0 |           81    |           81    |             6.8 |             6.8 |   1.15  |   2.01165  |
| (37, 'Ibrahim et al., 2005', 'pivot_shift_1')              | BPTB     | HT       |  40.5 |  45.5 |  5.5  |  8.5  |  0.99 |  1.54 |       22.3  |       22.3  |      3.94 |      3.84 |       40 |       46 | 40      |      46 |         0 |         0 |           81    |           81    |             6.8 |             6.8 |   0.684 |   0.595872 |
| (38, 'Iliopoulos et al., 2017', 'ikdc_subjective')         | BPTB     | HT       |  45   |  45   | 83.2  | 82.78 |  2.8  |  2.8  |       24.8  |       26.7  |      5    |      7.2  |       10 |       10 | 10.0    |      10 |         0 |         0 |            9    |            9    |             0.8 |             0.8 |   0.15  |   0.211    |
| (38, 'Iliopoulos et al., 2017', 'lysholm')                 | BPTB     | HT       |  45   |  45   | 95.8  | 94    |  2.4  |  3.6  |       24.8  |       26.7  |      5    |      7.2  |       10 |       10 | 10      |      10 |         0 |         0 |            9    |            9    |             0.8 |             0.8 |   0.588 |   0.215    |
| (41, 'Jansson et al., 2003', 'pivot_shift_2')              | BPTB     | HT       |  43.5 |  46.5 |  0.5  |  0.5  |  0.57 |  0.55 |      nan    |      nan    |    nan    |    nan    |      nan |      nan | nan     |     nan |       nan |       nan |           24    |           24    |             2   |             2   |   1.07  |   2.01122  |
| (41, 'Jansson et al., 2003', 'tegner')                     | BPTB     | HT       |  43   |  46   |  5.98 |  5.82 |  1.6  |  2.26 |      nan    |      nan    |    nan    |    nan    |      nan |      nan | nan     |     nan |       nan |       nan |           24    |           24    |             2   |             2   |   0.081 |   0.212    |
| (41, 'Jansson et al., 2003', 'instrumented_laxity')        | BPTB     | HT       |  43   |  46   |  1.7  |  1.2  |  2.63 |  2.65 |      nan    |      nan    |    nan    |    nan    |      nan |      nan | nan     |     nan |       nan |       nan |           24    |           24    |             2   |             2   |   0.189 |   0.213    |
| (41, 'Jansson et al., 2003', 'lachman_2')                  | BPTB     | HT       |  43.5 |  46.5 |  0.5  |  0.5  |  1.59 |  1.52 |      nan    |      nan    |    nan    |    nan    |      nan |      nan | nan     |     nan |       nan |       nan |           24    |           24    |             2   |             2   |   1.07  |   2.01122  |
| (41, 'Jansson et al., 2003', 'pivot_shift_1')              | BPTB     | HT       |  43.5 |  46.5 |  3.5  |  3.5  |  0.57 |  0.55 |      nan    |      nan    |    nan    |    nan    |      nan |      nan | nan     |     nan |       nan |       nan |           24    |           24    |             2   |             2   |   1.075 |   0.7872   |
| (41, 'Jansson et al., 2003', 'lachman_1')                  | BPTB     | HT       |  43.5 |  46.5 |  8.5  |  8.5  |  1.59 |  1.52 |      nan    |      nan    |    nan    |    nan    |      nan |      nan | nan     |     nan |       nan |       nan |           24    |           24    |             2   |             2   |   1.086 |   0.538685 |
| (43, 'Kautzner et al., 2015', 'instrumented_laxity')       | BPTB     | HT       |  74   |  73   |  0.6  |  0.5  |  0.21 |  0.42 |       26    |       26    |      6.28 |      6.29 |       74 |       73 | 0       |       0 |        74 |        73 |           24    |           24    |             2   |             2   |   0.302 |   0.166    |
| (43, 'Kautzner et al., 2015', 'graft_failure')             | BPTB     | HT       |  74.5 |  73.5 |  0.5  |  2.5  |  0.06 |  0.3  |       26    |       26    |      6.28 |      6.29 |       74 |       73 | 0       |       0 |        74 |        73 |           24    |           24    |             2   |             2   |   0.192 |   1.55808  |
| (43, 'Kautzner et al., 2015', 'lysholm')                   | BPTB     | HT       |  74   |  73   | 88    | 90    |  7.5  |  7.6  |       26    |       26    |      6.28 |      6.29 |       74 |       73 | 0       |       0 |        74 |        73 |           24    |           24    |             2   |             2   |   0.265 |   0.166    |
| (44, 'Komzak et al., 2022', 'lysholm')                     | BPTB     | QT       |  36   |  36   | 93.4  | 90.1  |  9.6  |  6.7  |       31    |       29.8  |      9.46 |      9.46 |       40 |       40 | 23      |      21 |        17 |        19 |           28    |           28    |             2.3 |             2.3 |   0.399 |   0.238    |
| (44, 'Komzak et al., 2022', 'ikdc_subjective')             | BPTB     | QT       |  36   |  36   | 74.2  | 78.3  | 10.2  | 10.5  |       31    |       29.8  |      9.46 |      9.46 |       40 |       40 | 23.0    |      21 |        17 |        19 |           28    |           28    |             2.3 |             2.3 |   0.396 |   0.238    |
| (45, 'Konrads et al., 2016', 'lysholm')                    | BPTB     | HT       |  24   |  23   | 92    | 91.8  |  8.99 |  9.33 |       29.8  |       29.8  |      6.33 |      6.33 |       31 |       31 | 22      |      23 |         9 |         8 |          120    |          120    |            10   |            10   |   0.022 |   0.292    |
| (45, 'Konrads et al., 2016', 'instrumented_laxity')        | BPTB     | HT       |  24   |  23   |  1.8  |  2.2  |  0.51 |  0.78 |       29.8  |       29.8  |      6.33 |      6.33 |       31 |       31 | 22      |      23 |         9 |         8 |          120    |          120    |            10   |            10   |   0.61  |   0.299    |
| (45, 'Konrads et al., 2016', 'tegner')                     | BPTB     | HT       |  24   |  23   |  5.9  |  5.1  |  1.28 |  1.04 |       29.8  |       29.8  |      6.33 |      6.33 |       31 |       31 | 22      |      23 |         9 |         8 |          120    |          120    |            10   |            10   |   0.684 |   0.301    |
| (45, 'Konrads et al., 2016', 'lachman_1')                  | BPTB     | HT       |  24.5 |  23.5 |  8.5  | 11.5  |  2.6  |  4.6  |       29.8  |       29.8  |      6.33 |      6.33 |       31 |       31 | 22      |      23 |         9 |         8 |          120    |          120    |            10   |            10   |   0.554 |   0.591977 |
| (45, 'Konrads et al., 2016', 'lachman_2')                  | BPTB     | HT       |  24.5 |  23.5 |  0.5  |  2.5  |  2.6  |  4.6  |       29.8  |       29.8  |      6.33 |      6.33 |       31 |       31 | 22      |      23 |         9 |         8 |          120    |          120    |            10   |            10   |   0.175 |   1.57775  |
| (46, 'Laoruengthana et al., 2009', 'pivot_shift_2')        | BPTB     | HT       |  15.5 |  13.5 |  0.5  |  0.5  |  0.75 |  0.82 |       29.93 |       28.65 |      8.91 |      9.27 |      nan |      nan | nan     |     nan |       nan |       nan |           19    |           19    |             1.6 |             1.6 |   0.867 |   2.03558  |
| (46, 'Laoruengthana et al., 2009', 'instrumented_laxity')  | BPTB     | HT       |  15   |  13   |  1.22 |  0.96 |  0.69 |  0.42 |       29.93 |       28.65 |      8.91 |      9.27 |      nan |      nan | nan     |     nan |       nan |       nan |           19    |           19    |             1.6 |             1.6 |   0.447 |   0.384    |
| (46, 'Laoruengthana et al., 2009', 'pivot_shift_1')        | BPTB     | HT       |  15.5 |  13.5 |  2.5  |  2.5  |  0.75 |  0.82 |       29.93 |       28.65 |      8.91 |      9.27 |      nan |      nan | nan     |     nan |       nan |       nan |           19    |           19    |             1.6 |             1.6 |   0.846 |   0.983785 |
| (47, 'Lawhorn et al., 2012', 'pivot_shift_1')              | HT       | TA       |  44.5 |  48.5 |  6.5  |  4.5  |  1.07 |  0.75 |       32    |       33    |      8.43 |      8.31 |       54 |       48 | 32      |      38 |        22 |        10 |           24    |           24    |             2   |             2   |   1.673 |   0.652006 |
| (47, 'Lawhorn et al., 2012', 'pivot_shift_2')              | HT       | TA       |  44.5 |  48.5 |  1.5  |  0.5  |  1.07 |  0.75 |       32    |       33    |      8.43 |      8.31 |       54 |       48 | 32      |      38 |        22 |        10 |           24    |           24    |             2   |             2   |   3.349 |   1.64644  |
| (49, 'Laxdal et al., 2005', 'graft_failure')               | BPTB     | HT       |  40.5 |  78.5 |  1.5  |  2.5  |  0.24 |  0.42 |       28    |       25    |      8.35 |      5.6  |      nan |      nan | nan     |     nan |       nan |       nan |           26    |           26    |             2.2 |             2.2 |   1.169 |   1.05141  |
| (49, 'Laxdal et al., 2005', 'lysholm')                     | BPTB     | HT       |  40   |  78   | 94    | 92    | 18.09 | 13.4  |       28    |       25    |      8.35 |      5.6  |      nan |      nan | nan     |     nan |       nan |       nan |           26    |           26    |             2.2 |             2.2 |   0.132 |   0.195    |
| (49, 'Laxdal et al., 2005', 'tegner')                      | BPTB     | HT       |  40   |  78   |  6    |  4    |  1.86 |  1.64 |       28    |       25    |      8.35 |      5.6  |      nan |      nan | nan     |     nan |       nan |       nan |           26    |           26    |             2.2 |             2.2 |   1.165 |   0.209    |
| (49, 'Laxdal et al., 2005', 'instrumented_laxity')         | BPTB     | HT       |  40   |  78   |  1    |  1.5  |  3.01 |  3.12 |       28    |       25    |      8.35 |      5.6  |      nan |      nan | nan     |     nan |       nan |       nan |           26    |           26    |             2.2 |             2.2 |   0.162 |   0.195    |
| (51, 'Li et al., 2015', 'lysholm')                         | HT       | TA       |  32   |  32   | 91.3  | 88.7  | 11.5  |  8.6  |       29.8  |       30.5  |      7.9  |      6.1  |       32 |       32 | 17      |      15 |        15 |        17 |           69.6  |           73.2  |             5.8 |             6.1 |   0.256 |   0.251    |
| (51, 'Li et al., 2015', 'lachman_1')                       | HT       | TA       |  32.5 |  32.5 |  6.5  | 13.5  |  1.41 |  4.02 |       29.8  |       30.5  |      7.9  |      6.1  |       32 |       32 | 17      |      15 |        15 |        17 |           69.6  |           73.2  |             5.8 |             6.1 |   0.352 |   0.564813 |
| (51, 'Li et al., 2015', 'pivot_shift_2')                   | HT       | TA       |  32.5 |  32.5 |  0.5  |  1.5  |  0.91 |  2.34 |       29.8  |       30.5  |      7.9  |      6.1  |       32 |       32 | 17      |      15 |        15 |        17 |           69.6  |           73.2  |             5.8 |             6.1 |   0.323 |   1.65232  |
| (51, 'Li et al., 2015', 'ikdc_subjective')                 | HT       | TA       |  32   |  32   | 87.5  | 83.8  |  3.2  |  6.9  |       29.8  |       30.5  |      7.9  |      6.1  |       32 |       32 | 17.0    |      15 |        15 |        17 |           69.6  |           73.2  |             5.8 |             6.1 |   0.688 |   0.258    |
| (51, 'Li et al., 2015', 'lachman_2')                       | HT       | TA       |  32.5 |  32.5 |  2.5  |  4.5  |  1.41 |  4.02 |       29.8  |       30.5  |      7.9  |      6.1  |       32 |       32 | 17      |      15 |        15 |        17 |           69.6  |           73.2  |             5.8 |             6.1 |   0.519 |   0.831426 |
| (51, 'Li et al., 2015', 'tegner')                          | HT       | TA       |  32   |  32   |  7.3  |  7    |  1.3  |  2.1  |       29.8  |       30.5  |      7.9  |      6.1  |       32 |       32 | 17      |      15 |        15 |        17 |           69.6  |           73.2  |             5.8 |             6.1 |   0.172 |   0.25     |
| (51, 'Li et al., 2015', 'instrumented_laxity')             | HT       | TA       |  32   |  32   |  2.1  |  3.5  |  1.6  |  1.2  |       29.8  |       30.5  |      7.9  |      6.1  |       32 |       32 | 17      |      15 |        15 |        17 |           69.6  |           73.2  |             5.8 |             6.1 |   0.99  |   0.265    |
| (51, 'Li et al., 2015', 'pivot_shift_1')                   | HT       | TA       |  32.5 |  32.5 |  4.5  |  9.5  |  0.91 |  2.34 |       29.8  |       30.5  |      7.9  |      6.1  |       32 |       32 | 17      |      15 |        15 |        17 |           69.6  |           73.2  |             5.8 |             6.1 |   0.389 |   0.637713 |
| (52, 'Lidén et al., 2007', 'tegner')                       | BPTB     | HT       |  30   |  32   |  5    |  6    |  1.72 |  1.69 |       28    |       29    |      8.37 |     10.35 |       34 |       37 | 23      |      26 |        11 |        11 |           87.11 |           87.05 |             7.3 |             7.3 |   0.587 |   0.26     |
| (52, 'Lidén et al., 2007', 'lysholm')                      | BPTB     | HT       |  30   |  32   | 81    | 90    | 18.38 | 12.1  |       28    |       29    |      8.37 |     10.35 |       34 |       37 | 23      |      26 |        11 |        11 |           87.11 |           87.05 |             7.3 |             7.3 |   0.582 |   0.26     |
| (52, 'Lidén et al., 2007', 'lachman_2')                    | BPTB     | HT       |  30.5 |  32.5 |  1.5  |  2.5  | 15.41 | 20.61 |       28    |       29    |      8.37 |     10.35 |       34 |       37 | 23      |      26 |        11 |        11 |           87.11 |           87.05 |             7.3 |             7.3 |   0.621 |   1.06512  |
| (52, 'Lidén et al., 2007', 'lachman_1')                    | BPTB     | HT       |  30.5 |  32.5 | 23.5  | 26.5  | 15.41 | 20.61 |       28    |       29    |      8.37 |     10.35 |       34 |       37 | 23      |      26 |        11 |        11 |           87.11 |           87.05 |             7.3 |             7.3 |   0.76  |   0.62435  |
| (52, 'Lidén et al., 2007', 'instrumented_laxity')          | BPTB     | HT       |  30   |  32   |  2.3  |  2.7  |  3.06 |  3.39 |       28    |       29    |      8.37 |     10.35 |       34 |       37 | 23      |      26 |        11 |        11 |           87.11 |           87.05 |             7.3 |             7.3 |   0.124 |   0.254    |
| (53, 'Lind et al., 2020', 'pivot_shift_2')                 | HT       | QT       |  49.5 |  47.5 |  0.5  |  0.5  |  2.12 |  1.49 |       27.1  |       27.2  |      6.1  |      6.2  |       49 |       50 | 25      |      29 |        24 |        21 |           24    |           24    |             2   |             2   |   0.959 |   2.01039  |
| (53, 'Lind et al., 2020', 'ikdc_subjective')               | HT       | QT       |  49   |  47   | 78    | 82    | 18    | 14    |       27.1  |       27.2  |      6.1  |      6.2  |       49 |       50 | 25.0    |      29 |        24 |        21 |           24    |           24    |             2   |             2   |   0.247 |   0.205    |
| (53, 'Lind et al., 2020', 'tegner')                        | HT       | QT       |  49   |  47   |  5.9  |  6    |  1.3  |  1.5  |       27.1  |       27.2  |      6.1  |      6.2  |       49 |       50 | 25      |      29 |        24 |        21 |           24    |           24    |             2   |             2   |   0.071 |   0.204    |
| (53, 'Lind et al., 2020', 'instrumented_laxity')           | HT       | QT       |  49   |  47   |  1.9  |  1.8  |  1.7  |  1    |       27.1  |       27.2  |      6.1  |      6.2  |       49 |       50 | 25      |      29 |        24 |        21 |           24    |           24    |             2   |             2   |   0.071 |   0.204    |
| (53, 'Lind et al., 2020', 'pivot_shift_1')                 | HT       | QT       |  49.5 |  47.5 | 11.5  |  8.5  |  2.12 |  1.49 |       27.1  |       27.2  |      6.1  |      6.2  |       49 |       50 | 25      |      29 |        24 |        21 |           24    |           24    |             2   |             2   |   1.389 |   0.506518 |
| (54, 'Lucidi et al., 2025', 'pivot_shift_2')               | BPTB     | HT       |  19.5 |  20.5 |  7.5  |  5.5  |  2.72 |  1.64 |       52.8  |       54.5  |      7.3  |      7.5  |      nan |      nan | nan     |     nan |       nan |       nan |          240    |          240    |            20   |            20   |   1.705 |   0.68202  |
| (54, 'Lucidi et al., 2025', 'lysholm')                     | BPTB     | HT       |  19   |  20   | 84.8  | 88.6  | 10.3  |  8.3  |       52.8  |       54.5  |      7.3  |      7.5  |      nan |      nan | nan     |     nan |       nan |       nan |          240    |          240    |            20   |            20   |   0.407 |   0.324    |
| (54, 'Lucidi et al., 2025', 'ikdc_subjective')             | BPTB     | HT       |  19   |  20   | 78.4  | 81.6  | 13.7  | 11.9  |       52.8  |       54.5  |      7.3  |      7.5  |      nan |      nan | nan     |     nan |       nan |       nan |          240    |          240    |            20   |            20   |   0.25  |   0.322    |
| (54, 'Lucidi et al., 2025', 'pivot_shift_1')               | BPTB     | HT       |  19.5 |  20.5 |  7.5  |  5.5  |  2.72 |  1.64 |       52.8  |       54.5  |      7.3  |      7.5  |      nan |      nan | nan     |     nan |       nan |       nan |          240    |          240    |            20   |            20   |   1.705 |   0.68202  |
| (54, 'Lucidi et al., 2025', 'instrumented_laxity')         | BPTB     | HT       |  19   |  20   | -0.1  |  1.5  |  1.6  |  1.2  |       52.8  |       54.5  |      7.3  |      7.5  |      nan |      nan | nan     |     nan |       nan |       nan |          240    |          240    |            20   |            20   |   1.136 |   0.347    |
| (54, 'Lucidi et al., 2025', 'tegner')                      | BPTB     | HT       |  19   |  20   |  3    |  4    |  1.08 |  0.8  |       52.8  |       54.5  |      7.3  |      7.5  |      nan |      nan | nan     |     nan |       nan |       nan |          240    |          240    |            20   |            20   |   1.056 |   0.343    |
| (54, 'Lucidi et al., 2025', 'graft_failure')               | BPTB     | HT       |  19.5 |  20.5 |  7.5  |  3.5  |  2.72 |  0.92 |       52.8  |       54.5  |      7.3  |      7.5  |      nan |      nan | nan     |     nan |       nan |       nan |          240    |          240    |            20   |            20   |   3.036 |   0.749136 |
| (55, 'Lui et al., 2012', 'lysholm')                        | BPTB     | HT       |  14   |  17   | 84    | 80    | 13.47 | 15.59 |       26.5  |       24.63 |      5    |      4.6  |       14 |       38 | 14      |      38 |         0 |         0 |           12    |           12    |             1   |             1   |   0.273 |   0.363    |
| (55, 'Lui et al., 2012', 'lachman_2')                      | BPTB     | HT       |  14.5 |  17.5 |  0.5  |  0.5  |  1.68 |  1.03 |       26.5  |       24.63 |      5    |      4.6  |       14 |       38 | 14      |      38 |         0 |         0 |           12    |           12    |             1   |             1   |   1.214 |   2.0323   |
| (55, 'Lui et al., 2012', 'instrumented_laxity')            | BPTB     | HT       |  14   |  17   |  3.7  |  3.1  |  1.4  |  1.6  |       26.5  |       24.63 |      5    |      4.6  |       14 |       38 | 14      |      38 |         0 |         0 |           12    |           12    |             1   |             1   |   0.396 |   0.365    |
| (55, 'Lui et al., 2012', 'lachman_1')                      | BPTB     | HT       |  14.5 |  17.5 |  4.5  |  6.5  |  1.68 |  1.03 |       26.5  |       24.63 |      5    |      4.6  |       14 |       38 | 14      |      38 |         0 |         0 |           12    |           12    |             1   |             1   |   0.762 |   0.752979 |
| (55, 'Lui et al., 2012', 'ikdc_subjective')                | BPTB     | HT       |  14   |  17   | 71.9  | 73.1  | 15.2  | 12.3  |       26.5  |       24.63 |      5    |      4.6  |       14 |       38 | 14.0    |      38 |         0 |         0 |           12    |           12    |             1   |             1   |   0.088 |   0.361    |
| (56, 'Lund et al., 2014', 'instrumented_laxity')           | BPTB     | QT       |  18   |  21   |  1.1  |  0.8  |  1.4  |  1.7  |       31    |       30    |      8    |      9    |       18 |       21 | 15      |      17 |         3 |         4 |           24    |           24    |             2   |             2   |   0.191 |   0.322    |
| (56, 'Lund et al., 2014', 'pivot_shift_1')                 | BPTB     | QT       |  18.5 |  21.5 |  3.5  |  8.5  |  0.99 |  3    |       31    |       30    |      8    |      9    |       18 |       21 | 15      |      17 |         3 |         4 |           24    |           24    |             2   |             2   |   0.357 |   0.739561 |
| (56, 'Lund et al., 2014', 'ikdc_subjective')               | BPTB     | QT       |  18   |  21   | 75    | 76    | 13    | 16    |       31    |       30    |      8    |      9    |       18 |       21 | 15.0    |      17 |         3 |         4 |           24    |           24    |             2   |             2   |   0.068 |   0.321    |
| (56, 'Lund et al., 2014', 'pivot_shift_2')                 | BPTB     | QT       |  18.5 |  21.5 |  3.5  |  8.5  |  0.99 |  3    |       31    |       30    |      8    |      9    |       18 |       21 | 15      |      17 |         3 |         4 |           24    |           24    |             2   |             2   |   0.357 |   0.739561 |
| (58, 'Malik et al., 2026', 'lachman_2')                    | HT       | PLT      |  25.5 |  25.5 |  3.5  |  2.5  |  0.8  |  0.54 |       28.12 |       27.84 |      5.21 |      4.92 |       25 |       25 | 18      |      17 |         7 |         8 |            6    |            6    |             0.5 |             0.5 |   1.464 |   0.88014  |
| (58, 'Malik et al., 2026', 'ikdc_subjective')              | HT       | PLT      |  25   |  25   | 78.9  | 82.6  |  7.4  |  6.3  |       28.12 |       27.84 |      5.21 |      4.92 |       25 |       25 | 18.0    |      17 |         7 |         8 |            6    |            6    |             0.5 |             0.5 |   0.538 |   0.288    |
| (58, 'Malik et al., 2026', 'lachman_1')                    | HT       | PLT      |  25.5 |  25.5 |  3.5  |  2.5  |  0.8  |  0.54 |       28.12 |       27.84 |      5.21 |      4.92 |       25 |       25 | 18      |      17 |         7 |         8 |            6    |            6    |             0.5 |             0.5 |   1.464 |   0.88014  |
| (58, 'Malik et al., 2026', 'lysholm')                      | HT       | PLT      |  25   |  25   | 85.1  | 89.4  |  6.4  |  5.8  |       28.12 |       27.84 |      5.21 |      4.92 |       25 |       25 | 18      |      17 |         7 |         8 |            6    |            6    |             0.5 |             0.5 |   0.704 |   0.292    |
| (58, 'Malik et al., 2026', 'pivot_shift_1')                | HT       | PLT      |  25.5 |  25.5 |  4.5  |  3.5  |  1.07 |  0.8  |       28.12 |       27.84 |      5.21 |      4.92 |       25 |       25 | 18      |      17 |         7 |         8 |            6    |            6    |             0.5 |             0.5 |   1.347 |   0.775248 |
| (58, 'Malik et al., 2026', 'tegner')                       | HT       | PLT      |  25   |  25   |  5.1  |  5.6  |  1    |  0.9  |       28.12 |       27.84 |      5.21 |      4.92 |       25 |       25 | 18      |      17 |         7 |         8 |            6    |            6    |             0.5 |             0.5 |   0.526 |   0.288    |
| (58, 'Malik et al., 2026', 'instrumented_laxity')          | HT       | PLT      |  25   |  25   |  2.1  |  1.8  |  1    |  0.9  |       28.12 |       27.84 |      5.21 |      4.92 |       25 |       25 | 18      |      17 |         7 |         8 |            6    |            6    |             0.5 |             0.5 |   0.315 |   0.285    |
| (58, 'Malik et al., 2026', 'pivot_shift_2')                | HT       | PLT      |  25.5 |  25.5 |  4.5  |  3.5  |  1.07 |  0.8  |       28.12 |       27.84 |      5.21 |      4.92 |       25 |       25 | 18      |      17 |         7 |         8 |            6    |            6    |             0.5 |             0.5 |   1.347 |   0.775248 |
| (62, 'Matsumoto et al., 2006', 'ikdc_subjective')          | BPTB     | HT       |  37   |  35   | 84.3  | 86.8  | 11    |  8.3  |       23.7  |       24.4  |      7.7  |      9.7  |       37 |       35 | 21.0    |      20 |        16 |        15 |           87    |           80.7  |             7.3 |             6.7 |   0.256 |   0.237    |
| (62, 'Matsumoto et al., 2006', 'instrumented_laxity')      | BPTB     | HT       |  37   |  35   |  1.2  |  1.7  |  2.1  |  1.4  |       23.7  |       24.4  |      7.7  |      9.7  |       37 |       35 | 21      |      20 |        16 |        15 |           87    |           80.7  |             7.3 |             6.7 |   0.279 |   0.237    |
| (66, 'Mohtadi et al., 2016', 'graft_failure')              | BPTB     | HT       | 110.5 | 110.5 | 19.5  | 24.5  |  2.25 |  2.99 |       28.7  |       28.4  |      9.9  |      9.83 |      110 |      220 | 63      |     120 |        47 |       100 |           24    |           24    |             2   |             2   |   0.752 |   0.338696 |
| (68, 'Moraiti et al., 2010', 'ikdc_subjective')            | BPTB     | HT       |   6   |   6   | 92    | 90    |  7    | 10    |       23    |       24    |      2    |      3    |        6 |        6 | 6.0     |       6 |         0 |         0 |           21    |           25    |             1.8 |             2.1 |   0.232 |   0.58     |
| (68, 'Moraiti et al., 2010', 'lysholm')                    | BPTB     | HT       |   6   |   6   | 96    | 95    |  3    |  5    |       23    |       24    |      2    |      3    |        6 |        6 | 6       |       6 |         0 |         0 |           21    |           25    |             1.8 |             2.1 |   0.243 |   0.58     |
| (70, 'Noh et al., 2011', 'lysholm')                        | HT       | AT       |  33   |  32   | 98    | 99    |  3.61 |  3.63 |       25.81 |       25.55 |      7.46 |      8.47 |       33 |       32 | 30      |      26 |         3 |         6 |           28.1  |           31.6  |             2.3 |             2.6 |   0.276 |   0.249    |
| (70, 'Noh et al., 2011', 'tegner')                         | HT       | AT       |  33   |  32   |  6    |  6    |  0.96 |  1.21 |       25.81 |       25.55 |      7.46 |      8.47 |       33 |       32 | 30      |      26 |         3 |         6 |           28.1  |           31.6  |             2.3 |             2.6 |   0     |   0.248    |
| (70, 'Noh et al., 2011', 'lachman_2')                      | HT       | AT       |  33.5 |  32.5 |  1.5  |  3.5  |  1.38 |  3.1  |       25.81 |       25.55 |      7.46 |      8.47 |       33 |       32 | 30      |      26 |         3 |         6 |           28.1  |           31.6  |             2.3 |             2.6 |   0.388 |   1.00902  |
| (70, 'Noh et al., 2011', 'pivot_shift_1')                  | HT       | AT       |  33.5 |  32.5 |  6.5  | 11.5  |  1.38 |  1.41 |       25.81 |       25.55 |      7.46 |      8.47 |       33 |       32 | 30      |      26 |         3 |         6 |           28.1  |           31.6  |             2.3 |             2.6 |   0.44  |   0.57049  |
| (70, 'Noh et al., 2011', 'pivot_shift_2')                  | HT       | AT       |  33.5 |  32.5 |  1.5  |  3.5  |  1.38 |  1.41 |       25.81 |       25.55 |      7.46 |      8.47 |       33 |       32 | 30      |      26 |         3 |         6 |           28.1  |           31.6  |             2.3 |             2.6 |   0.388 |   1.00902  |
| (70, 'Noh et al., 2011', 'lachman_1')                      | HT       | AT       |  33.5 |  32.5 |  6.5  |  6.5  |  1.38 |  3.1  |       25.81 |       25.55 |      7.46 |      8.47 |       33 |       32 | 30      |      26 |         3 |         6 |           28.1  |           31.6  |             2.3 |             2.6 |   0.963 |   0.619024 |
| (73, 'Papalia et al., 2015', 'ikdc_subjective')            | BPTB     | HT       |  20   |  20   | 86.9  | 90.97 | 11.7  |  5.1  |       24.7  |       25.4  |    nan    |    nan    |       20 |       20 | 0.0     |       0 |        20 |        20 |            6    |            6    |             0.5 |             0.5 |   0.451 |   0.32     |
| (73, 'Papalia et al., 2015', 'lysholm')                    | BPTB     | HT       |  20   |  20   | 87.4  | 93.17 | 13.8  | 10.3  |       24.7  |       25.4  |    nan    |    nan    |       20 |       20 | 0       |       0 |        20 |        20 |            6    |            6    |             0.5 |             0.5 |   0.474 |   0.321    |
| (76, 'Pinczewski et al., 2007', 'instrumented_laxity')     | BPTB     | HT       |  54   |  58   |  1.2  |  1.6  |  1.69 |  1.55 |       25    |       24    |      5.49 |      7.92 |       90 |       90 | 48      |      47 |        42 |        43 |          120    |          120    |            10   |            10   |   0.247 |   0.19     |
| (76, 'Pinczewski et al., 2007', 'pivot_shift_2')           | BPTB     | HT       |  53.5 |  58.5 |  0.5  |  0.5  |  0.83 |  1.3  |       25    |       24    |      5.49 |      7.92 |       90 |       90 | 48      |      47 |        42 |        43 |          120    |          120    |            10   |            10   |   1.094 |   2.00901  |
| (76, 'Pinczewski et al., 2007', 'pivot_shift_1')           | BPTB     | HT       |  53.5 |  58.5 |  5.5  |  8.5  |  0.83 |  1.3  |       25    |       24    |      5.49 |      7.92 |       90 |       90 | 48      |      47 |        42 |        43 |          120    |          120    |            10   |            10   |   0.674 |   0.583351 |
| (76, 'Pinczewski et al., 2007', 'graft_failure')           | BPTB     | HT       |  90.5 |  90.5 |  7.5  | 12.5  |  0.86 |  1.52 |       25    |       24    |      5.49 |      7.92 |       90 |       90 | 48      |      47 |        42 |        43 |          120    |          120    |            10   |            10   |   0.564 |   0.488059 |
| (76, 'Pinczewski et al., 2007', 'lachman_2')               | BPTB     | HT       |  53.5 |  58.5 |  0.5  |  1.5  |  1.78 |  2.07 |       25    |       24    |      5.49 |      7.92 |       90 |       90 | 48      |      47 |        42 |        43 |          120    |          120    |            10   |            10   |   0.358 |   1.6441   |
| (76, 'Pinczewski et al., 2007', 'lachman_1')               | BPTB     | HT       |  53.5 |  58.5 | 10.5  | 12.5  |  1.78 |  2.07 |       25    |       24    |      5.49 |      7.92 |       90 |       90 | 48      |      47 |        42 |        43 |          120    |          120    |            10   |            10   |   0.899 |   0.46929  |
| (77, 'Popovic et al., 2024', 'graft_failure')              | BPTB     | HT       |  38.5 |  33.5 |  3.5  |  4.5  |  0.62 |  0.89 |       27    |       27    |      6.82 |      6.3  |        0 |      nan | `       |     nan |       nan |       nan |          228    |          228    |            19   |            19   |   0.644 |   0.755639 |
| (77, 'Popovic et al., 2024', 'lachman_1')                  | BPTB     | HT       |  38.5 |  33.5 |  0.5  |  5.5  |  0.08 |  0.09 |       27    |       27    |      6.82 |      6.3  |        0 |      nan | `       |     nan |       nan |       nan |          228    |          228    |            19   |            19   |   0.067 |   1.49795  |
| (77, 'Popovic et al., 2024', 'lachman_2')                  | BPTB     | HT       |  38.5 |  33.5 |  9.5  |  5.5  |  0.08 |  0.09 |       27    |       27    |      6.82 |      6.3  |        0 |      nan | `       |     nan |       nan |       nan |          228    |          228    |            19   |            19   |   1.668 |   0.597728 |
| (77, 'Popovic et al., 2024', 'instrumented_laxity')        | BPTB     | HT       |  38   |  33   |  1.3  |  1.5  |  2.2  |  3.3  |       27    |       27    |      6.82 |      6.3  |        0 |      nan | `       |     nan |       nan |       nan |          228    |          228    |            19   |            19   |   0.072 |   0.238    |
| (77, 'Popovic et al., 2024', 'lysholm')                    | BPTB     | HT       |  38   |  33   | 82    | 85    | 15.45 | 14.67 |       27    |       27    |      6.82 |      6.3  |        0 |      nan | `       |     nan |       nan |       nan |          228    |          228    |            19   |            19   |   0.199 |   0.239    |
| (77, 'Popovic et al., 2024', 'pivot_shift_2')              | BPTB     | HT       |  38.5 |  33.5 |  1.5  |  3.5  |  0.62 |  0.09 |       27    |       27    |      6.82 |      6.3  |        0 |      nan | `       |     nan |       nan |       nan |          228    |          228    |            19   |            19   |   0.347 |   1.00635  |
| (77, 'Popovic et al., 2024', 'tegner')                     | BPTB     | HT       |  38   |  33   |  4    |  5    |  1.64 |  2.16 |       27    |       27    |      6.82 |      6.3  |        0 |      nan | `       |     nan |       nan |       nan |          228    |          228    |            19   |            19   |   0.527 |   0.242    |
| (77, 'Popovic et al., 2024', 'pivot_shift_1')              | BPTB     | HT       |  38.5 |  33.5 |  1.5  |  3.5  |  0.62 |  0.09 |       27    |       27    |      6.82 |      6.3  |        0 |      nan | `       |     nan |       nan |       nan |          228    |          228    |            19   |            19   |   0.347 |   1.00635  |
| (82, 'Sajovic et al., 2018', 'pivot_shift_2')              | BPTB     | HT       |  24.5 |  24.5 |  0.5  |  0.5  |  1.77 |  1.42 |       45.5  |       42.5  |      8.7  |      7.5  |       24 |       24 | 15      |      13 |         9 |        11 |          204    |          204    |            17   |            17   |   1     |   2.02073  |
| (82, 'Sajovic et al., 2018', 'lachman_2')                  | BPTB     | HT       |  24.5 |  24.5 |  0.5  |  0.5  |  1.42 |  1.42 |       45.5  |       42.5  |      8.7  |      7.5  |       24 |       24 | 15      |      13 |         9 |        11 |          204    |          204    |            17   |            17   |   1     |   2.02073  |
| (82, 'Sajovic et al., 2018', 'pivot_shift_1')              | BPTB     | HT       |  24.5 |  24.5 |  5.5  |  6.5  |  1.77 |  1.42 |       45.5  |       42.5  |      8.7  |      7.5  |       24 |       24 | 15      |      13 |         9 |        11 |          204    |          204    |            17   |            17   |   0.802 |   0.666222 |
| (82, 'Sajovic et al., 2018', 'lachman_1')                  | BPTB     | HT       |  24.5 |  24.5 |  5.5  |  5.5  |  1.42 |  1.42 |       45.5  |       42.5  |      8.7  |      7.5  |       24 |       24 | 15      |      13 |         9 |        11 |          204    |          204    |            17   |            17   |   1     |   0.684762 |
| (84, 'Sinding et al., 2020', 'ikdc_subjective')            | HT       | QT       |  43   |  42   | 76    | 76    | 15    | 17    |       28.3  |       28.7  |      6.2  |      6.4  |       43 |       42 | 23.0    |      25 |        20 |        17 |           12.5  |           12.4  |             1   |             1   |   0     |   0.217    |
| (85, 'Smith et al., 2020', 'graft_failure')                | BPTB     | HT       |  32.5 |  32.5 |  0.5  |  2.5  |  0.09 |  0.47 |       17.7  |       17.7  |      2.4  |      2.4  |       32 |       32 | 15      |      13 |        17 |        19 |           24    |           24    |             2   |             2   |   0.188 |   1.5699   |
| (85, 'Smith et al., 2020', 'ikdc_subjective')              | BPTB     | HT       |  32   |  32   | 89.4  | 94.8  |  9.2  |  6.4  |       17.7  |       17.7  |      2.4  |      2.4  |       32 |       32 | 15.0    |      13 |        17 |        19 |           24    |           24    |             2   |             2   |   0.681 |   0.257    |
| (85, 'Smith et al., 2020', 'instrumented_laxity')          | BPTB     | HT       |  32   |  32   |  0    |  0.3  |  0.8  |  0.7  |       17.7  |       17.7  |      2.4  |      2.4  |       32 |       32 | 15      |      13 |        17 |        19 |           24    |           24    |             2   |             2   |   0.399 |   0.253    |
| (87, 'Sonnery-cottet et al., 2026', 'graft_failure')       | BPTB     | HT       | 273.5 | 283.5 | 12.5  | 12.5  |  0.79 |  0.78 |       24.8  |       25.2  |      4.4  |      4.6  |      283 |      273 | 214     |     233 |        69 |        40 |           61.5  |           63.9  |             5.1 |             5.3 |   1.038 |   0.409294 |
| (87, 'Sonnery-cottet et al., 2026', 'tegner')              | BPTB     | HT       | 296   | 297   |  6.6  |  6.7  |  1.6  |  1.7  |       24.8  |       25.2  |      4.4  |      4.6  |      283 |      273 | 214     |     233 |        69 |        40 |           61.5  |           63.9  |             5.1 |             5.3 |   0.061 |   0.082    |
| (87, 'Sonnery-cottet et al., 2026', 'instrumented_laxity') | BPTB     | HT       | 296   | 297   |  1.2  |  0.9  |  1.4  |  1    |       24.8  |       25.2  |      4.4  |      4.6  |      283 |      273 | 214     |     233 |        69 |        40 |           61.5  |           63.9  |             5.1 |             5.3 |   0.247 |   0.082    |
| (87, 'Sonnery-cottet et al., 2026', 'ikdc_subjective')     | BPTB     | HT       | 296   | 297   | 89.4  | 91    | 10.9  | 10.3  |       24.8  |       25.2  |      4.4  |      4.6  |      283 |      273 | 214.0   |     233 |        69 |        40 |           61.5  |           63.9  |             5.1 |             5.3 |   0.151 |   0.082    |
| (87, 'Sonnery-cottet et al., 2026', 'lysholm')             | BPTB     | HT       | 296   | 297   | 89    | 91.1  | 12.4  | 11.9  |       24.8  |       25.2  |      4.4  |      4.6  |      283 |      273 | 214     |     233 |        69 |        40 |           61.5  |           63.9  |             5.1 |             5.3 |   0.173 |   0.082    |
| (89, 'Tang et al., 2023', 'ikdc_subjective')               | HT       | QT       |  16   |  17   | 90.13 | 87.12 |  6.99 |  6.61 |       28.31 |       28.06 |      8.55 |      6.24 |       16 |       17 | 13.0    |      17 |         3 |         0 |           24    |           24    |             2   |             2   |   0.443 |   0.353    |
| (89, 'Tang et al., 2023', 'lysholm')                       | HT       | QT       |  16   |  17   | 90.06 | 90    |  7.33 |  4.26 |       28.31 |       28.06 |      8.55 |      6.24 |       16 |       17 | 13      |      17 |         3 |         0 |           24    |           24    |             2   |             2   |   0.01  |   0.348    |
| (90, 'Taylor et al., 2009', 'graft_failure')               | BPTB     | HT       |  32.5 |  32.5 |  3.5  |  4.5  |  0.68 |  0.91 |       21.7  |       22.1  |      4.6  |      6.53 |       32 |       32 | 25      |      28 |         7 |         4 |           32.4  |           38.4  |             2.7 |             3.2 |   0.751 |   0.760351 |
| (90, 'Taylor et al., 2009', 'tegner')                      | BPTB     | HT       |  20   |  24   |  6.8  |  5.3  |  3.27 |  3.58 |       21.7  |       22.1  |      4.6  |      6.53 |       32 |       32 | 25      |      28 |         7 |         4 |           32.4  |           38.4  |             2.7 |             3.2 |   0.436 |   0.306    |
| (90, 'Taylor et al., 2009', 'lysholm')                     | BPTB     | HT       |  21   |  23   | 90.3  | 90.4  | 12.19 | 12.75 |       21.7  |       22.1  |      4.6  |      6.53 |       32 |       32 | 25      |      28 |         7 |         4 |           32.4  |           38.4  |             2.7 |             3.2 |   0.008 |   0.302    |
| (93, 'Vilchez-Cavazos et al., 2020', 'ikdc_subjective')    | HT       | QT       |  14   |  14   | 90    | 88.17 |  1.76 |  2.49 |       24.46 |       24.46 |      8.24 |      9.06 |       15 |       14 | 12.0    |      11 |         3 |         3 |           12    |           12    |             1   |             1   |   0.849 |   0.396    |
| (93, 'Vilchez-cavazos et al., 2020', 'lysholm')            | BPTB     | QT       |  14   |  14   | 96    | 95.33 |  2.93 |  2.64 |      nan    |       24.46 |    nan    |      9.06 |      nan |       14 | nan     |      11 |       nan |         3 |          nan    |           12    |           nan   |             1   |   0.24  |   0.379    |
| (95, 'Webster et al., 2001', 'instrumented_laxity')        | BPTB     | HT       |  23   |  31   |  1.1  |  1.7  |  0.65 |  0.97 |       26    |       27    |      6.3  |      6.1  |       28 |       33 | 20      |      23 |         8 |        10 |           24    |           24    |             2   |             2   |   0.706 |   0.284    |
| (96, 'Wipfler et al., 2011', 'graft_failure')              | BPTB     | HT       |  29.5 |  25.5 |  3.5  |  3.5  |  0.72 |  0.8  |       29.87 |       34.23 |      7.3  |      9.25 |       31 |       31 | 19      |      18 |        12 |        13 |          108    |          108    |             9   |             9   |   0.846 |   0.809534 |
| (96, 'Wipfler et al., 2011', 'ikdc_subjective')            | BPTB     | HT       |  29   |  25   | 86.72 | 92.95 |  2.27 |  1.99 |       29.87 |       34.23 |      7.3  |      9.25 |       31 |       31 | 19.0    |      18 |        12 |        13 |          108    |          108    |             9   |             9   |   2.904 |   0.394    |
| (96, 'Wipfler et al., 2011', 'tegner')                     | BPTB     | HT       |  29   |  25   |  6.2  |  6.14 |  0.35 |  0.37 |       29.87 |       34.23 |      7.3  |      9.25 |       31 |       31 | 19      |      18 |        12 |        13 |          108    |          108    |             9   |             9   |   0.167 |   0.273    |
| (96, 'Wipfler et al., 2011', 'instrumented_laxity')        | BPTB     | HT       |  29   |  25   |  0.9  |  0.64 |  0.27 |  0.36 |       29.87 |       34.23 |      7.3  |      9.25 |       31 |       31 | 19      |      18 |        12 |        13 |          108    |          108    |             9   |             9   |   0.826 |   0.285    |
| (96, 'Wipfler et al., 2011', 'lysholm')                    | BPTB     | HT       |  29   |  25   | 87.28 | 91.82 |  1.76 |  1.71 |       29.87 |       34.23 |      7.3  |      9.25 |       31 |       31 | 19      |      18 |        12 |        13 |          108    |          108    |             9   |             9   |   2.614 |   0.374    |
| (97, 'Yadav et al., 2025', 'pivot_shift_2')                | HT       | QT       |  36.5 |  33.5 |  1.5  |  0.5  |  1.07 |  0.09 |       30.06 |       28.73 |      8.87 |      8.8  |       36 |       33 | 32      |      30 |         4 |         3 |           12    |           12    |             1   |             1   |   2.829 |   1.65092  |
| (97, 'Yadav et al., 2025', 'graft_failure')                | HT       | QT       |  36.5 |  33.5 |  1.5  |  0.5  |  0.26 |  0.09 |       30.06 |       28.73 |      8.87 |      8.8  |       36 |       33 | 32      |      30 |         4 |         3 |           12    |           12    |             1   |             1   |   2.829 |   1.65092  |
| (97, 'Yadav et al., 2025', 'lysholm')                      | HT       | QT       |  36   |  33   | 92.4  | 88.2  |  0    |  0    |       30.06 |       28.73 |      8.87 |      8.8  |       36 |       33 | 32      |      30 |         4 |         3 |           12    |           12    |             1   |             1   | inf     | inf        |
| (97, 'Yadav et al., 2025', 'instrumented_laxity')          | HT       | QT       |  36   |  33   |  3.77 |  2.73 |  2.01 |  1.64 |       30.06 |       28.73 |      8.87 |      8.8  |       36 |       33 | 32      |      30 |         4 |         3 |           12    |           12    |             1   |             1   |   0.564 |   0.246    |
| (97, 'Yadav et al., 2025', 'pivot_shift_1')                | HT       | QT       |  36.5 |  33.5 |  5.5  |  0.5  |  1.07 |  0.09 |       30.06 |       28.73 |      8.87 |      8.8  |       36 |       33 | 32      |      30 |         4 |         3 |           12    |           12    |             1   |             1   |  11.71  |   1.49813  |
| (98, 'Yoo et al., 2017', 'lachman_2')                      | HT       | TA       |  68.5 |  64.5 |  4.5  |  6.5  |  0.58 |  0.9  |       30    |       24    |      9.96 |      8.35 |       68 |       64 | 61      |      59 |         7 |         5 |           32.8  |           34.5  |             2.7 |             2.9 |   0.627 |   0.63948  |
| (98, 'Yoo et al., 2017', 'pivot_shift_2')                  | HT       | TA       |  68.5 |  64.5 |  2.5  |  3.5  |  0.31 |  0.46 |       30    |       24    |      9.96 |      8.35 |       68 |       64 | 61      |      59 |         7 |         5 |           32.8  |           34.5  |             2.7 |             2.9 |   0.66  |   0.846912 |
| (98, 'Yoo et al., 2017', 'tegner')                         | HT       | TA       |  68   |  64   |  5    |  5    |  1.48 |  1.07 |       30    |       24    |      9.96 |      8.35 |       68 |       64 | 61      |      59 |         7 |         5 |           32.8  |           34.5  |             2.7 |             2.9 |   0     |   0.174    |
| (98, 'Yoo et al., 2017', 'lachman_1')                      | HT       | TA       |  68.5 |  64.5 | 36.5  | 35.5  |  0.58 |  0.9  |       30    |       24    |      9.96 |      8.35 |       68 |       64 | 61      |      59 |         7 |         5 |           32.8  |           34.5  |             2.7 |             2.9 |   0.932 |   0.34828  |
| (98, 'Yoo et al., 2017', 'pivot_shift_1')                  | HT       | TA       |  68.5 |  64.5 | 26.5  | 33.5  |  0.31 |  0.46 |       30    |       24    |      9.96 |      8.35 |       68 |       64 | 61      |      59 |         7 |         5 |           32.8  |           34.5  |             2.7 |             2.9 |   0.584 |   0.351645 |
| (98, 'Yoo et al., 2017', 'lysholm')                        | HT       | TA       |  68   |  64   | 96    | 93    |  6.99 |  5.78 |       30    |       24    |      9.96 |      8.35 |       68 |       64 | 61      |      59 |         7 |         5 |           32.8  |           34.5  |             2.7 |             2.9 |   0.466 |   0.177    |
| (100, 'Zaffagnini et al., 2011', 'instrumented_laxity')    | BPTB     | HT       |  40   |  39   |  0.4  |  1.1  |  0.6  |  1.9  |       26    |       27    |      9.5  |      9    |       39 |       40 | 20      |      22 |        19 |        18 |          103.2  |          103.2  |             8.6 |             8.6 |   0.5   |   0.229    |
| (100, 'Zaffagnini et al., 2011', 'ikdc_subjective')        | BPTB     | HT       |  40   |  39   | 82    | 88    | 20    |  9    |       26    |       27    |      9.5  |      9    |       39 |       40 | 20.0    |      22 |        19 |        18 |          103.2  |          103.2  |             8.6 |             8.6 |   0.385 |   0.227    |